# Motor Exercise 5 — fitting a local straight-line model

Plot settled speed across the complete tested range, fit one straight line inside a chosen moving and unsaturated region, and check one command that was not used for fitting.

Start with the supplied synthetic example so that every cell runs before you have collected data. The example demonstrates the plotting route; it is not evidence about your robot and is not a result you should expect to reproduce. When you are ready, change only the settings in **Use the example or your own data** and run the notebook again.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
rng = np.random.default_rng(2026)


## 1. Use the example or your own data

Leave `USE_EXAMPLE_DATA` set to `True` on your first run. To use your measurements, upload the CSV, set it to `False`, and enter the filename. This is the main cell you need to edit.

Expected CSV columns: `wheel`, `direction`, `PWM`, `trial_num`, and `settled_speed_cps`.


In [ ]:
USE_EXAMPLE_DATA = True
CSV_FILENAME = "motor_exercise05_gain.csv"

print("Using:", "synthetic example" if USE_EXAMPLE_DATA else CSV_FILENAME)


## 2. Create the small synthetic example

The synthetic response is approximately linear only through part of the tested range. Its values are illustrative.


In [ ]:
example_rows = []
for pwm in range(10, 171, 10):
    typical_speed = np.clip(
        8.2 * pwm - 155 + 0.0072 * (pwm - 90) ** 2,
        0,
        930,
    )
    for trial_num in range(1, 6):
        example_rows.append({
            "wheel": "left",
            "direction": "forward",
            "PWM": pwm,
            "trial_num": trial_num,
            "settled_speed_cps": typical_speed + rng.normal(0, 10),
        })

example_data = pd.DataFrame(example_rows)


## 3. Load and preview the selected data

This is where your uploaded CSV enters the notebook. Check the first rows before continuing: column names, units and labels should match the exercise.


In [ ]:
if USE_EXAMPLE_DATA:
    data = example_data.copy()
else:
    data = pd.read_csv(CSV_FILENAME)

data.head()


## 4. Choose one case and the fitting region

Use your deadband and saturation evidence to choose the bounds. Reserve one measured command before fitting; it will test the resulting line.


In [ ]:
SELECTED_WHEEL = "left"
SELECTED_DIRECTION = "forward"
FIT_PWM_MIN = 40
FIT_PWM_MAX = 130
HELD_BACK_PWM = 90

selected = data.loc[
    (data["wheel"] == SELECTED_WHEEL)
    & (data["direction"] == SELECTED_DIRECTION)
].copy()


## 5. Plot every observation before fitting


In [ ]:
sns.stripplot(
    data=selected,
    x="PWM",
    y="settled_speed_cps",
    jitter=0.16,
    alpha=0.5,
    native_scale=True,
)
plt.axvspan(FIT_PWM_MIN, FIT_PWM_MAX, color="tab:blue", alpha=0.08)
plt.axvline(HELD_BACK_PWM, color="tab:red", linestyle="--")
plt.title("Complete response; shaded region selected for fitting")
plt.xlabel("Requested PWM")
plt.ylabel("Settled encoder speed (counts/s)")
plt.show()


## 6. Summarise repeats, fit the line and report its equation

The line is fitted to condition means inside the selected range, except
for the held-back command. The repeat spread remains visible in the table.

We write the fitted line as

$$\hat{\omega} = G u + c$$

where $u$ is requested PWM, $G$ is the fitted gain (the slope), and $c$
is the fitted intercept. `np.polyfit(..., 1)` returns these two
coefficients in the order **slope, then intercept**.


In [ ]:
response_summary = (
    selected.groupby("PWM", as_index=False)
    .agg(
        measured_speed_cps=("settled_speed_cps", "mean"),
        repeat_spread_cps=("settled_speed_cps", "std"),
    )
    .sort_values("PWM")
)

fit_rows = response_summary.loc[
    response_summary["PWM"].between(FIT_PWM_MIN, FIT_PWM_MAX)
    & (response_summary["PWM"] != HELD_BACK_PWM)
]

gain, intercept = np.polyfit(
    fit_rows["PWM"],
    fit_rows["measured_speed_cps"],
    1,
)
response_summary["predicted_speed_cps"] = (
    gain * response_summary["PWM"] + intercept
)
response_summary["residual_cps"] = (
    response_summary["measured_speed_cps"]
    - response_summary["predicted_speed_cps"]
)

coefficient_table = pd.DataFrame([
    {
        "coefficient": "gain",
        "symbol": "G",
        "value": gain,
        "units": "counts/s per PWM unit",
        "meaning": "change in predicted speed for one PWM unit",
    },
    {
        "coefficient": "intercept",
        "symbol": "c",
        "value": intercept,
        "units": "counts/s",
        "meaning": "predicted speed where the fitted line reaches PWM 0",
    },
])

print("Response summary, including repeat spread:")
print(response_summary.to_string(index=False))
print()
print(f"Fitted equation: omega_hat = {gain:.3f} * PWM {intercept:+.3f}")
coefficient_table


The intercept is where the fitted line would reach PWM 0. If PWM 0 is outside your selected fitting region, this is an extrapolation; it should not automatically be interpreted as the motor's physical speed at zero command.


## 7. Plot the fitted line and residuals


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

axes[0].errorbar(
    response_summary["PWM"],
    response_summary["measured_speed_cps"],
    yerr=response_summary["repeat_spread_cps"],
    marker="o",
    linestyle="none",
    capsize=3,
    label="measured mean ± repeat SD",
)
axes[0].plot(
    response_summary["PWM"],
    response_summary["predicted_speed_cps"],
    color="black",
    label="local linear prediction",
)
axes[0].set(title="Local straight-line model", ylabel="Speed (counts/s)")
axes[0].legend()

sns.scatterplot(
    data=response_summary,
    x="PWM",
    y="residual_cps",
    ax=axes[1],
)
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set(
    title="Measured minus predicted speed",
    xlabel="Requested PWM",
    ylabel="Residual (counts/s)",
)
plt.tight_layout()
plt.show()


## 8. Inspect the command kept out of fitting


In [ ]:
held_back_result = response_summary.loc[
    response_summary["PWM"] == HELD_BACK_PWM,
    [
        "PWM",
        "measured_speed_cps",
        "predicted_speed_cps",
        "residual_cps",
        "repeat_spread_cps",
    ],
]

held_back_result


## What to notice

- Do the residuals look small and unstructured inside the fitting region?
- How does the held-back error compare with repeated measurement spread?
- Where does the line visibly stop describing the complete response?
- State the wheel, direction, PWM range and speed units with the fitted gain.
